# FitCheck 빠른 사이즈 비교 — DM-VTON

**런타임 → 런타임 유형 변경 → GPU(T4)**를 선택하고 셀을 순서대로 실행하세요.
화면에는 전신 사진, 기존 옷 치수, 새 옷 사이즈표, 상품 사진·합성 결과가 나옵니다.
기존 치수와 사진 속 옷은 같은 옷이어야 합니다. 상품 사진은 옷 한 벌이 잘 보이는 사진 또는 투명 PNG를 사용하세요.

DM-VTON 가중치는 약 113MB, 자동 옷 영역 분할 모델은 약 244MB를 처음 한 번 받습니다.
현재 코랩의 PyTorch를 그대로 사용하며 구형 CuPy 설치는 하지 않습니다. 모델 구조와 공식 가중치를 유지하고 correlation 연산을 PyTorch로 실행합니다.
논문의 23.3ms와 다른 실행 조건이므로 화면에 표시되는 실제 처리 시간을 확인하세요.

사진이 같으면 기본 합성을 재사용해 치수 변경을 비교합니다. 사이즈 결과는 자동 영역에 치수 비율을 적용한 실험이며 실제 착용 핏 검증은 별도입니다.

In [ ]:
# 1. GPU 확인
import subprocess
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)

In [ ]:
# 2. 현재 FitCheck 코드 받기
import os
import sys
from pathlib import Path
APP = '/content/fitcheck-dmvton'
REPOSITORY = 'https://github.com/eunzae02-glitch/virtual-fitting-app-master.git'
if not Path(APP).exists():
    subprocess.run(['git', 'clone', '--branch', 'main', REPOSITORY, APP], check=True)
else:
    origin = subprocess.check_output(['git', '-C', APP, 'remote', 'get-url', 'origin'], text=True).strip()
    if origin.rstrip('/').removesuffix('.git') != REPOSITORY.removesuffix('.git'):
        raise RuntimeError('APP에 다른 저장소가 있습니다. APP를 새 경로로 바꿔주세요.')
    subprocess.run(['git', '-C', APP, 'pull', '--ff-only'], check=True)
if not (Path(APP) / 'scripts/debug_dmvton.py').is_file():
    raise RuntimeError('PC 변경사항을 GitHub에 Push한 뒤 다시 실행해주세요.')
os.environ['DMVTON_REPO'] = '/content/DM-VTON'
os.environ['DMVTON_WEIGHTS'] = '/content/dmvton-weights'
print('코드 준비 완료')

In [ ]:
# 3. 웹 화면·추론·자동 옷 영역 추출 패키지 설치
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{APP}/requirements_dmvton.txt'], check=True)
print('설치 완료')

In [ ]:
# 4. 공식 DM-VTON 소스·가중치 두 개와 영역 분할 캐시 준비
subprocess.run([sys.executable, '-u', f'{APP}/scripts/setup_dmvton.py'], check=True)

In [ ]:
# 5. 단순 사이즈 비교 화면 — 출력의 gradio.live 링크를 여세요.
# 사용하는 동안 실행 상태를 유지하고, 새 서버를 열기 전 기존 실행 셀을 중지하세요.
server = subprocess.Popen([sys.executable, '-u', f'{APP}/scripts/debug_dmvton.py', '--share'])
try:
    server.wait()
finally:
    if server.poll() is None:
        server.terminate()
        try:
            server.wait(timeout=10)
        except subprocess.TimeoutExpired:
            server.kill()
            server.wait()